# MagiAttention FFA vs FlexAttention-CuteDSL

Notebook analogue of `tests/test_attn/test_ffa_vs_flex_attention_hd192.py`.

**Correctness:** MagiAttention FFA (`magi_attention.functional.flex_flash_attn_func`) vs
Magi **FlexAttention-CuteDSL** (`magi_attention.kernel.cutedsl.flex_flash_attn_func` with
`TorchFlexAttnArgs.mask_mod`). Falls back to PyTorch `flex_attention` if CuteDSL is unavailable.

**Throughput:** theoretical FLOPs / measured time (TFLOP/s) for Magi FFA vs FlexAttention-CuteDSL
(and PyTorch FlexAttention) across `head_dim`.

Masks (same as the pytest):
`full`, `causal`, `inv_causal`, `bi_causal`, `sliding_window_causal`, `sliding_window_full`,
`varlen_full`, `varlen_causal`.

Recommended kernel: `dmikhaylov-k6v_magi` (Magi FFA + FlexAttention). Magi CuteDSL needs
`nvidia-cutlass-dsl>=4.4` (e.g. `karaev-magi-cuda12.8`).


In [1]:
from __future__ import annotations

import math
import os
import sys
import traceback
from collections.abc import Callable
from functools import partial
from typing import Any

import pandas as pd
import torch
from torch.nn.attention.flex_attention import create_block_mask, flex_attention

# Repo root on PYTHONPATH when launched from elsewhere
_REPO = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if os.path.isdir(os.path.join(_REPO, "magi_attention")) and _REPO not in sys.path:
    sys.path.insert(0, _REPO)

from magi_attention.api.functools import infer_attn_mask_from_sliding_window
from magi_attention.benchmarking import do_bench_flops
from magi_attention.common.enum import AttnMaskType
from magi_attention.common.range import AttnRange
from magi_attention.common.ranges import AttnRanges
from magi_attention.functional import flex_flash_attn_func as magi_ffa
from magi_attention.utils import str2seed

import torch._functorch.config as _functorch_config

_functorch_config.donated_buffer = False

DEVICE = torch.device("cuda", torch.cuda.current_device())
print("torch", torch.__version__, "| device", DEVICE, "|", torch.cuda.get_device_name(DEVICE))

# Magi FlexAttention-CuteDSL (optional)
CUTEDSL_OK = False
cute_ffa = None
TorchFlexAttnArgs = None
MT_MAP = None
try:
    from magi_attention.kernel.cutedsl import (
        MT_MAP as _MT_MAP,
    )
    from magi_attention.kernel.cutedsl import (
        TorchFlexAttnArgs as _TorchFlexAttnArgs,
    )
    from magi_attention.kernel.cutedsl import (
        flex_flash_attn_func as _cute_ffa,
    )

    cute_ffa = _cute_ffa
    TorchFlexAttnArgs = _TorchFlexAttnArgs
    MT_MAP = _MT_MAP
    CUTEDSL_OK = True
    print("FlexAttention-CuteDSL: available (magi_attention.kernel.cutedsl)")
except Exception as e:
    print("FlexAttention-CuteDSL: UNAVAILABLE —", type(e).__name__, str(e)[:200])
    print("Falling back to torch.nn.attention.flex_attention as the baseline.")

/home/jovyan/dmikhaylov/src/MagiAttention/magi_attention/__init__.py:30: UserWarning: Failed to import magi_attn_ext extension module. Please make sure MagiAttention is properly installed. Original error message: cannot import name 'magi_attn_ext' from partially initialized module 'magi_attention' (most likely due to a circular import) (/home/jovyan/dmikhaylov/src/MagiAttention/magi_attention/__init__.py)
  warnings.warn(
/home/jovyan/dmikhaylov/src/MagiAttention/magi_attention/__init__.py:39: UserWarning: Failed to import magi_attn_comm extension module. Please make sure MagiAttention is properly installed. Original error message: cannot import name 'magi_attn_comm' from partially initialized module 'magi_attention' (most likely due to a circular import) (/home/jovyan/dmikhaylov/src/MagiAttention/magi_attention/__init__.py)
  warnings.warn(
/home/jovyan/dmikhaylov/src/MagiAttention/magi_attention/__init__.py:46: UserWarning: You are using magi_attention without installing it. This may

torch 2.10.0+cu130 | device cuda:0 | NVIDIA H100 80GB HBM3
FlexAttention-CuteDSL: UNAVAILABLE — ImportError cannot import name 'PipelineClcFetchAsync' from 'cutlass.pipeline' (/home/jovyan/dmikhaylov/envs/k6v_magi/lib/python3.13/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/pipeline/__init__.py)
Falling back to torch.nn.attention.flex_attention as the baseline.


## Helpers (shared with the pytest)

In [2]:
MaskMod = Callable[[Any, Any, Any, Any], torch.Tensor]

_FLEX_BLOCK_SIZE = 32
SEED = 42

MASK_NAMES = [
    "full",
    "causal",
    "inv_causal",
    "bi_causal",
    "sliding_window_causal",
    "sliding_window_full",
    "varlen_full",
    "varlen_causal",
]


def _thd_to_bhsd(x: torch.Tensor) -> torch.Tensor:
    return x.transpose(0, 1).unsqueeze(0).contiguous()


def _bhsd_to_thd(x: torch.Tensor) -> torch.Tensor:
    return x.squeeze(0).transpose(0, 1).contiguous()


def _thd_to_bshd(x: torch.Tensor) -> torch.Tensor:
    """(S, H, D) -> (1, S, H, D) for Magi CuteDSL dense path."""
    return x.unsqueeze(0).contiguous()


def _bshd_to_thd(x: torch.Tensor) -> torch.Tensor:
    return x.squeeze(0).contiguous()


def _ranges_to_tensors(q_ranges, k_ranges, attn_type_map, device):
    q_t = torch.as_tensor(q_ranges, device=device, dtype=torch.int32)
    k_t = torch.as_tensor(k_ranges, device=device, dtype=torch.int32)
    a_t = torch.as_tensor(attn_type_map, device=device, dtype=torch.int32)
    return q_t, k_t, a_t


def _err(actual: torch.Tensor, expected: torch.Tensor) -> tuple[float, float]:
    a = actual.detach().float()
    e = expected.detach().float()
    abs_err = float((a - e).abs().max())
    rel_err = abs_err / max(float(e.abs().max()), 1e-6)
    return abs_err, rel_err


def _mask_full(b, h, q_idx, kv_idx):
    return q_idx == q_idx


def _mask_causal(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx


def _mask_inv_causal(b, h, q_idx, kv_idx):
    return q_idx <= kv_idx


def _mask_bi_causal(b, h, q_idx, kv_idx):
    return q_idx == kv_idx


def _mask_sliding_window_causal(b, h, q_idx, kv_idx, window_left: int):
    return (q_idx >= kv_idx) & (q_idx - kv_idx <= window_left)


def _mask_sliding_window_full(b, h, q_idx, kv_idx, window: int):
    return (q_idx - kv_idx).abs() <= window


def _mask_varlen_full(b, h, q_idx, kv_idx, document_id: torch.Tensor):
    return document_id[q_idx] == document_id[kv_idx]


def _mask_varlen_causal(b, h, q_idx, kv_idx, document_id: torch.Tensor):
    return (document_id[q_idx] == document_id[kv_idx]) & (q_idx >= kv_idx)


def _packed_docs(seqlen: int) -> list[tuple[int, int]]:
    return [
        (0, seqlen // 4),
        (seqlen // 4, 3 * seqlen // 4),
        (3 * seqlen // 4, seqlen),
    ]


def _document_id(seqlen: int, device: torch.device) -> torch.Tensor:
    doc_id = torch.zeros(seqlen, device=device, dtype=torch.int32)
    for i, (start, end) in enumerate(_packed_docs(seqlen)):
        doc_id[start:end] = i
    return doc_id


def build_mask_case(name: str, seqlen: int, device: torch.device):
    """Return Magi (q_ranges, k_ranges, attn_type_map) + Flex mask_mod."""
    if name == "full":
        return (
            *_ranges_to_tensors(
                [[0, seqlen]],
                [[0, seqlen]],
                [AttnMaskType.FULL.to_int_type()],
                device,
            ),
            _mask_full,
        )
    if name == "causal":
        return (
            *_ranges_to_tensors(
                [[0, seqlen]],
                [[0, seqlen]],
                [AttnMaskType.CAUSAL.to_int_type()],
                device,
            ),
            _mask_causal,
        )
    if name == "inv_causal":
        return (
            *_ranges_to_tensors(
                [[0, seqlen]],
                [[0, seqlen]],
                [AttnMaskType.INVCAUSAL.to_int_type()],
                device,
            ),
            _mask_inv_causal,
        )
    if name == "bi_causal":
        return (
            *_ranges_to_tensors(
                [[0, seqlen]],
                [[0, seqlen]],
                [AttnMaskType.BICAUSAL.to_int_type()],
                device,
            ),
            _mask_bi_causal,
        )
    if name == "sliding_window_causal":
        window_left = min(64, seqlen - 1)
        q_obj, k_obj, mask_types = infer_attn_mask_from_sliding_window(
            q_range=AttnRange(0, seqlen),
            k_range=AttnRange(0, seqlen),
            window_size=(window_left, 0),
        )
        return (
            *_ranges_to_tensors(
                q_obj.to_naive_ranges(),
                k_obj.to_naive_ranges(),
                [m.to_int_type() for m in mask_types],
                device,
            ),
            partial(_mask_sliding_window_causal, window_left=window_left),
        )
    if name == "sliding_window_full":
        window = min(32, seqlen - 1)
        q_obj, k_obj, mask_types = infer_attn_mask_from_sliding_window(
            q_range=AttnRange(0, seqlen),
            k_range=AttnRange(0, seqlen),
            window_size=(window, window),
        )
        return (
            *_ranges_to_tensors(
                q_obj.to_naive_ranges(),
                k_obj.to_naive_ranges(),
                [m.to_int_type() for m in mask_types],
                device,
            ),
            partial(_mask_sliding_window_full, window=window),
        )
    if name in ("varlen_full", "varlen_causal"):
        docs = _packed_docs(seqlen)
        causal = name == "varlen_causal"
        attn_type_map = [
            AttnMaskType.CAUSAL.to_int_type() if causal else AttnMaskType.FULL.to_int_type()
            for _ in docs
        ]
        document_id = _document_id(seqlen, device)
        mask_mod = partial(
            _mask_varlen_causal if causal else _mask_varlen_full,
            document_id=document_id,
        )
        return (
            *_ranges_to_tensors(
                [[a, b] for a, b in docs],
                [[a, b] for a, b in docs],
                attn_type_map,
                device,
            ),
            mask_mod,
        )
    raise ValueError(name)


def _wrap_cutedsl_mask_mod(mask_mod: MaskMod):
    """Adapt torch Flex mask_mod(b,h,q,kv) -> CuteDSL (b,h,q,kv,seqlen_info,aux)."""

    def _mod(b, h, q_idx, kv_idx, seqlen_info=None, aux_tensors=None):
        return mask_mod(b, h, q_idx, kv_idx)

    return _mod


def calculate_attn_flops(
    q_ranges: AttnRanges,
    k_ranges: AttnRanges,
    attn_mask_type: list,
    total_seqlen_q: int,
    num_heads_q: int,
    head_dim: int,
) -> dict[str, float]:
    """Attention FLOPs for full/causal square ranges (Magi bench formula, no TE dep)."""
    area = 0.0
    for qr, kr, mt in zip(
        q_ranges.to_naive_ranges(), k_ranges.to_naive_ranges(), attn_mask_type
    ):
        sq = qr[1] - qr[0]
        sk = kr[1] - kr[0]
        is_causal = mt in (AttnMaskType.CAUSAL, AttnMaskType.CAUSAL.to_int_type())
        if is_causal and sq == sk:
            area += sq * (sq + 1) / 2.0
        else:
            area += float(sq * sk)
    flops_fwd = 4.0 * area * num_heads_q * head_dim
    flops_bwd = flops_fwd * 2.5
    return {"fwd": flops_fwd, "bwd": flops_bwd, "1f1b": flops_fwd + flops_bwd}



## Runners

In [3]:
def run_magi_ffa(q, k, v, q_ranges, k_ranges, attn_type_map, softmax_scale):
    out, _ = magi_ffa(
        q,
        k,
        v,
        q_ranges=q_ranges,
        k_ranges=k_ranges,
        attn_type_map=attn_type_map,
        softmax_scale=softmax_scale,
    )
    return out


def run_torch_flex(q_bhsd, k_bhsd, v_bhsd, mask_mod, softmax_scale, compiled=False):
    seqlen = q_bhsd.shape[-2]
    block_mask = create_block_mask(
        mask_mod,
        B=None,
        H=None,
        Q_LEN=seqlen,
        KV_LEN=seqlen,
        device=q_bhsd.device,
        BLOCK_SIZE=_FLEX_BLOCK_SIZE,
    )
    fn = torch.compile(flex_attention) if compiled else flex_attention
    return fn(q_bhsd, k_bhsd, v_bhsd, block_mask=block_mask, scale=softmax_scale)


def run_flex_cutedsl(q_bshd, k_bshd, v_bshd, mask_mod, softmax_scale):
    """Magi FlexAttention-CuteDSL dense path with programmable mask_mod."""
    if not CUTEDSL_OK:
        raise RuntimeError("Magi CuteDSL not available in this env")
    flex_args = TorchFlexAttnArgs(mask_mod=_wrap_cutedsl_mask_mod(mask_mod))
    out, _ = cute_ffa(
        q_bshd,
        k_bshd,
        v_bshd,
        mask_types=MT_MAP.full,
        softmax_scale=softmax_scale,
        flex_attn_args=flex_args,
    )
    return out


def fwd_bwd_pair(
    *,
    baseline: str,
    q0: torch.Tensor,
    k0: torch.Tensor,
    v0: torch.Tensor,
    do: torch.Tensor,
    q_ranges,
    k_ranges,
    attn_type_map,
    mask_mod,
    softmax_scale: float,
):
    """Run Magi FFA and baseline; return dict of thd tensors {out,dq,dk,dv} x {magi,base}."""
    # Magi FFA (thd)
    q_m = q0.clone().detach().requires_grad_(True)
    k_m = k0.clone().detach().requires_grad_(True)
    v_m = v0.clone().detach().requires_grad_(True)
    out_m = run_magi_ffa(q_m, k_m, v_m, q_ranges, k_ranges, attn_type_map, softmax_scale)
    out_m.backward(do)
    magi = {
        "out": out_m.detach(),
        "dq": q_m.grad.detach(),
        "dk": k_m.grad.detach(),
        "dv": v_m.grad.detach(),
    }

    if baseline == "flex_cutedsl":
        q_b = _thd_to_bshd(q0).detach().requires_grad_(True)
        k_b = _thd_to_bshd(k0).detach().requires_grad_(True)
        v_b = _thd_to_bshd(v0).detach().requires_grad_(True)
        out_b = run_flex_cutedsl(q_b, k_b, v_b, mask_mod, softmax_scale)
        out_b.backward(_thd_to_bshd(do))
        base = {
            "out": _bshd_to_thd(out_b.detach()),
            "dq": _bshd_to_thd(q_b.grad.detach()),
            "dk": _bshd_to_thd(k_b.grad.detach()),
            "dv": _bshd_to_thd(v_b.grad.detach()),
        }
    elif baseline == "torch_flex":
        q_b = _thd_to_bhsd(q0).detach().requires_grad_(True)
        k_b = _thd_to_bhsd(k0).detach().requires_grad_(True)
        v_b = _thd_to_bhsd(v0).detach().requires_grad_(True)
        out_b = run_torch_flex(q_b, k_b, v_b, mask_mod, softmax_scale, compiled=False)
        out_b.backward(_thd_to_bhsd(do))
        base = {
            "out": _bhsd_to_thd(out_b.detach()),
            "dq": _bhsd_to_thd(q_b.grad.detach()),
            "dk": _bhsd_to_thd(k_b.grad.detach()),
            "dv": _bhsd_to_thd(v_b.grad.detach()),
        }
    else:
        raise ValueError(baseline)

    return magi, base

## 1) Correctness table — fwd / bwd diffs vs FlexAttention-CuteDSL

Prefer Magi CuteDSL as baseline; if it is missing or fails, fall back to PyTorch `flex_attention` (eager).

After the run, the next cell prints a plain-text table **and writes readable files** under `exps/attn/cutedsl/results/`:

| file | content |
|------|---------|
| `ffa_vs_flex_correctness.md` | one markdown table per `seqlen` |
| `ffa_vs_flex_correctness.html` | open in browser (best for scrolling) |
| `ffa_vs_flex_correctness.csv` | wide pivot |
| `ffa_vs_flex_correctness_long.csv` | raw long form |



In [4]:
# Config — keep modest for interactive runs; widen as needed
CORR_HEAD_DIM = 192
CORR_SEQLENS = [128, 256]
CORR_NUM_HEADS = [2]
CORR_DTYPE = torch.bfloat16

PREFERRED_BASELINE = "flex_cutedsl" if CUTEDSL_OK else "torch_flex"
print("Preferred baseline:", PREFERRED_BASELINE)

rows = []
for mask_name in MASK_NAMES:
    for seqlen in CORR_SEQLENS:
        for num_heads in CORR_NUM_HEADS:
            torch.manual_seed(SEED + seqlen + num_heads + str2seed(mask_name) % 997)
            softmax_scale = 1.0 / math.sqrt(CORR_HEAD_DIM)
            q0 = torch.randn(
                seqlen, num_heads, CORR_HEAD_DIM, device=DEVICE, dtype=CORR_DTYPE
            )
            k0 = torch.randn(
                seqlen, num_heads, CORR_HEAD_DIM, device=DEVICE, dtype=CORR_DTYPE
            )
            v0 = torch.randn(
                seqlen, num_heads, CORR_HEAD_DIM, device=DEVICE, dtype=CORR_DTYPE
            )
            do = torch.randn_like(q0)
            q_ranges, k_ranges, attn_type_map, mask_mod = build_mask_case(
                mask_name, seqlen, DEVICE
            )

            baseline_used = PREFERRED_BASELINE
            err_msg = ""
            try:
                magi, base = fwd_bwd_pair(
                    baseline=baseline_used,
                    q0=q0,
                    k0=k0,
                    v0=v0,
                    do=do,
                    q_ranges=q_ranges,
                    k_ranges=k_ranges,
                    attn_type_map=attn_type_map,
                    mask_mod=mask_mod,
                    softmax_scale=softmax_scale,
                )
            except Exception as e:
                if baseline_used != "torch_flex":
                    baseline_used = "torch_flex"
                    try:
                        magi, base = fwd_bwd_pair(
                            baseline=baseline_used,
                            q0=q0,
                            k0=k0,
                            v0=v0,
                            do=do,
                            q_ranges=q_ranges,
                            k_ranges=k_ranges,
                            attn_type_map=attn_type_map,
                            mask_mod=mask_mod,
                            softmax_scale=softmax_scale,
                        )
                        err_msg = f"cutedsl failed → torch_flex ({type(e).__name__})"
                    except Exception as e2:
                        err_msg = f"FAIL: {type(e2).__name__}: {e2}"
                        magi = base = None
                else:
                    err_msg = f"FAIL: {type(e).__name__}: {e}"
                    magi = base = None

            if magi is None:
                rows.append(
                    {
                        "mask": mask_name,
                        "seqlen": seqlen,
                        "heads": num_heads,
                        "hd": CORR_HEAD_DIM,
                        "baseline": baseline_used,
                        "tensor": "-",
                        "max_abs": float("nan"),
                        "max_rel": float("nan"),
                        "note": err_msg,
                    }
                )
                continue

            for tname in ("out", "dq", "dk", "dv"):
                abs_err, rel_err = _err(magi[tname], base[tname])
                rows.append(
                    {
                        "mask": mask_name,
                        "seqlen": seqlen,
                        "heads": num_heads,
                        "hd": CORR_HEAD_DIM,
                        "baseline": baseline_used,
                        "tensor": tname,
                        "max_abs": abs_err,
                        "max_rel": rel_err,
                        "note": err_msg,
                    }
                )
            print(f"ok  {mask_name:24s} S={seqlen} baseline={baseline_used} {err_msg}")

corr_df = pd.DataFrame(rows)
print(f"correctness rows: {len(corr_df)}")




Preferred baseline: torch_flex


/home/jovyan/dmikhaylov/envs/k6v_magi/lib/python3.13/site-packages/torch/nn/attention/flex_attention.py:1624: UserWarning: flex_attention called without torch.compile() - this will use an unfused implementation that materializes the full scores matrix instead of generating a fused kernel.

SOLUTION: Use torch.compile(flex_attention)(...)

If you want to debug your score_mod/mask_mod, you can set:
torch.nn.attention.flex_attention._FLEX_ATTENTION_DISABLE_COMPILE_DEBUG = True

This will allow you to use print statements or breakpoints. Note: This doesn't work with the backwards pass and may produce incorrect results.
  _warn_once(


ok  full                     S=128 baseline=torch_flex 
ok  full                     S=256 baseline=torch_flex 
ok  causal                   S=128 baseline=torch_flex 
ok  causal                   S=256 baseline=torch_flex 
ok  inv_causal               S=128 baseline=torch_flex 
ok  inv_causal               S=256 baseline=torch_flex 
ok  bi_causal                S=128 baseline=torch_flex 
ok  bi_causal                S=256 baseline=torch_flex 
ok  sliding_window_causal    S=128 baseline=torch_flex 
ok  sliding_window_causal    S=256 baseline=torch_flex 
ok  sliding_window_full      S=128 baseline=torch_flex 
ok  sliding_window_full      S=256 baseline=torch_flex 
ok  varlen_full              S=128 baseline=torch_flex 
ok  varlen_full              S=256 baseline=torch_flex 
ok  varlen_causal            S=128 baseline=torch_flex 
ok  varlen_causal            S=256 baseline=torch_flex 


,mask,seqlen,heads,hd,baseline,tensor,max_abs,max_rel,note
0,full,128,2,192,torch_flex,out,0.003906,0.005495,
1,full,128,2,192,torch_flex,dq,0.003906,0.004831,
2,full,128,2,192,torch_flex,dk,0.003906,0.004255,
3,full,128,2,192,torch_flex,dv,0.001953,0.001838,
4,full,256,2,192,torch_flex,out,0.001953,0.003448,
...,...,...,...,...,...,...,...,...,...
59,varlen_causal,128,2,192,torch_flex,dv,0.000000,0.000000,
60,varlen_causal,256,2,192,torch_flex,out,0.015625,0.004237,
61,varlen_causal,256,2,192,torch_flex,dq,0.009766,0.003834,
62,varlen_causal,256,2,192,torch_flex,dk,0.015625,0.005464,


In [ ]:
from pathlib import Path

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 24)
pd.set_option("display.float_format", lambda x: f"{x:.3e}")

ok = corr_df.dropna(subset=["max_abs", "max_rel"]).copy()
long = ok.melt(
    id_vars=["mask", "seqlen", "tensor"],
    value_vars=["max_abs", "max_rel"],
    var_name="err",
    value_name="val",
)
long["err"] = long["err"].map({"max_abs": "abs", "max_rel": "rel"})
tensors = ["out", "dq", "dk", "dv"]
seqlens = sorted(ok["seqlen"].unique().tolist())
wide = (
    long.pivot_table(
        index="mask",
        columns=["seqlen", "tensor", "err"],
        values="val",
        aggfunc="max",
    )
    .reindex(index=MASK_NAMES)
    .reindex(columns=pd.MultiIndex.from_product([seqlens, tensors, ["abs", "rel"]]))
)

# Flatten columns for readable CSV / markdown: "S128_out_abs"
flat = wide.copy()
flat.columns = [f"S{s}_{t}_{e}" for s, t, e in flat.columns]
flat = flat.reset_index()

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / "ffa_vs_flex_correctness.csv"
md_path = out_dir / "ffa_vs_flex_correctness.md"
html_path = out_dir / "ffa_vs_flex_correctness.html"
long_csv_path = out_dir / "ffa_vs_flex_correctness_long.csv"

flat.to_csv(csv_path, index=False, float_format="%.6e")
corr_df.to_csv(long_csv_path, index=False, float_format="%.6e")

# Markdown: one section per seqlen (shorter lines, easier to read)
md_lines = [
    "# MagiFFA vs baseline — correctness errors",
    "",
    f"- baseline(s): `{corr_df.groupby('baseline').size().to_dict()}`",
    "",
]
for s in seqlens:
    cols = ["mask"] + [c for c in flat.columns if c.startswith(f"S{s}_")]
    sub = flat[cols].copy()
    sub.columns = ["mask"] + [c[len(f"S{s}_"):] for c in cols[1:]]
    md_lines.append(f"## seqlen = {s}")
    md_lines.append("")
    try:
        md_lines.append(sub.to_markdown(index=False, floatfmt=".3e"))
    except Exception:
        md_lines.append("```")
        md_lines.append(sub.to_string(index=False))
        md_lines.append("```")
    md_lines.append("")
md_path.write_text("\n".join(md_lines), encoding="utf-8")

# Standalone HTML (open in browser — full width, no Jupyter truncation)
sections = []
for s in seqlens:
    cols = ["mask"] + [c for c in flat.columns if c.startswith(f"S{s}_")]
    sub = flat[cols].copy()
    sub.columns = ["mask"] + [c[len(f"S{s}_"):] for c in cols[1:]]
    sections.append(f"<h2>seqlen = {s}</h2>")
    sections.append(sub.to_html(index=False, float_format=lambda x: f"{x:.3e}"))
html_path.write_text(
    "<!DOCTYPE html><html><head><meta charset='utf-8'>"
    "<title>FFA vs Flex correctness</title>"
    "<style>"
    "body{font-family:ui-monospace,Menlo,monospace;margin:16px;}"
    "table{border-collapse:collapse;font-size:13px;}"
    "th,td{border:1px solid #ccc;padding:4px 10px;text-align:right;white-space:nowrap;}"
    "th:first-child,td:first-child{text-align:left;position:sticky;left:0;background:#fff;}"
    "th{background:#f3f3f3;}"
    "h2{margin-top:24px;}"
    "</style></head><body>"
    "<h1>MagiFFA vs baseline — abs / rel</h1>"
    + "".join(sections)
    + "</body></html>",
    encoding="utf-8",
)

# Notebook: plain text (always fully visible) + paths
print("=== MagiFFA vs baseline (abs / rel) ===")
for s in seqlens:
    cols = ["mask"] + [c for c in flat.columns if c.startswith(f"S{s}_")]
    sub = flat[cols].copy()
    sub.columns = ["mask"] + [c[len(f"S{s}_"):] for c in cols[1:]]
    print(f"\n--- seqlen = {s} ---")
    print(sub.to_string(index=False))

print("\nbaselines used:", corr_df.groupby("baseline").size().to_dict())
print("saved:")
print(" ", csv_path.resolve())
print(" ", md_path.resolve())
print(" ", html_path.resolve())
print(" ", long_csv_path.resolve())



## 2) Throughput (TFLOP/s) vs seqlen / head_dim

Target shapes:
- `seqlen ∈ {1024, 2048}`
- `num_heads = 128`
- `(head_dim_q, head_dim_k, head_dim_v) ∈ {(192, 192, 128), (192, 192, 192)}`

Units (Magi `run_benchmark_simple.py`):
- `ms` — from `do_bench_flops()["flops"]` (**milliseconds**)
- `work_tflops` — theoretical work in TFLOPs: `2 * area * H * (dqk + dv) / 1e12`
- `tflops` — throughput: `work_FLOPs / ms * 1e-9` (**TFLOP/s**)

Compiled Flex uses `BLOCK_SIZE=128`.


In [ ]:
FLOP_SEQLENS = [1024, 2048]
FLOP_HEADS = 128
# (dq, dk, dv) — Magi / Flex both support dv != dqk (e.g. DeepSeek 192/128)
FLOP_HEAD_DIMS = [(192, 192, 128), (192, 192, 192)]
FLOP_MASKS = ["full", "causal"]
FLOP_DIRS = ["fwd", "bwd"]
FLOP_DTYPE = torch.bfloat16
WARMUP, REPS = 10, 30
_FLEX_BENCH_BLOCK = 128


def calculate_attn_flops(
    q_ranges,
    k_ranges,
    attn_mask_type,
    total_seqlen_q: int,
    num_heads_q: int,
    head_dim_qk: int,
    head_dim_v: int,
) -> dict[str, float]:
    """Fwd FLOPs = 2*area*H*dqk (QK) + 2*area*H*dv (PV). Bwd = 2.5x fwd."""
    area = 0.0
    for qr, kr, mt in zip(
        q_ranges.to_naive_ranges(), k_ranges.to_naive_ranges(), attn_mask_type
    ):
        sq = qr[1] - qr[0]
        sk = kr[1] - kr[0]
        is_causal = mt in (AttnMaskType.CAUSAL, AttnMaskType.CAUSAL.to_int_type())
        if is_causal and sq == sk:
            area += sq * (sq + 1) / 2.0
        else:
            area += float(sq * sk)
    flops_fwd = 2.0 * area * num_heads_q * (head_dim_qk + head_dim_v)
    return {"fwd": flops_fwd, "bwd": flops_fwd * 2.5, "1f1b": flops_fwd * 3.5}


_flex_compiled = torch.compile(flex_attention)


def _attn_flops(
    mask_name: str,
    seqlen: int,
    num_heads: int,
    head_dim_qk: int,
    head_dim_v: int,
) -> dict[str, float]:
    causal = mask_name == "causal"
    return calculate_attn_flops(
        q_ranges=AttnRanges.from_ranges([[0, seqlen]]),
        k_ranges=AttnRanges.from_ranges([[0, seqlen]]),
        attn_mask_type=[AttnMaskType.CAUSAL if causal else AttnMaskType.FULL],
        total_seqlen_q=seqlen,
        num_heads_q=num_heads,
        head_dim_qk=head_dim_qk,
        head_dim_v=head_dim_v,
    )


def _bench_impl(
    impl: str,
    mask_name: str,
    direction: str,
    seqlen: int,
    num_heads: int,
    head_dim_qk: int,
    head_dim_v: int,
):
    softmax_scale = 1.0 / math.sqrt(head_dim_qk)
    q_ranges, k_ranges, attn_type_map, mask_mod = build_mask_case(
        mask_name, seqlen, DEVICE
    )
    q0 = torch.randn(seqlen, num_heads, head_dim_qk, device=DEVICE, dtype=FLOP_DTYPE)
    k0 = torch.randn(seqlen, num_heads, head_dim_qk, device=DEVICE, dtype=FLOP_DTYPE)
    v0 = torch.randn(seqlen, num_heads, head_dim_v, device=DEVICE, dtype=FLOP_DTYPE)

    if impl == "magi_ffa":
        q = q0.clone().detach().requires_grad_(direction == "bwd")
        k = k0.clone().detach().requires_grad_(direction == "bwd")
        v = v0.clone().detach().requires_grad_(direction == "bwd")

        def fwd():
            return run_magi_ffa(
                q, k, v, q_ranges, k_ranges, attn_type_map, softmax_scale
            )

    elif impl == "flex_cutedsl":
        q = _thd_to_bshd(q0).detach().requires_grad_(direction == "bwd")
        k = _thd_to_bshd(k0).detach().requires_grad_(direction == "bwd")
        v = _thd_to_bshd(v0).detach().requires_grad_(direction == "bwd")

        def fwd():
            return run_flex_cutedsl(q, k, v, mask_mod, softmax_scale)

    elif impl == "torch_flex":
        q = _thd_to_bhsd(q0).detach().requires_grad_(direction == "bwd")
        k = _thd_to_bhsd(k0).detach().requires_grad_(direction == "bwd")
        v = _thd_to_bhsd(v0).detach().requires_grad_(direction == "bwd")
        block_mask = create_block_mask(
            mask_mod,
            B=None,
            H=None,
            Q_LEN=seqlen,
            KV_LEN=seqlen,
            device=DEVICE,
            BLOCK_SIZE=_FLEX_BENCH_BLOCK,
        )

        def fwd():
            return _flex_compiled(
                q, k, v, block_mask=block_mask, scale=softmax_scale
            )

    else:
        raise ValueError(impl)

    out = fwd()
    torch.cuda.synchronize()

    if direction == "fwd":
        def fn():
            fwd()

        grad_to_none = None
    else:
        do = torch.randn_like(out)
        out = fwd()
        torch.cuda.synchronize()

        def fn():
            out.backward(do, retain_graph=True)

        grad_to_none = [q, k, v]

    ms = do_bench_flops(
        fn,
        warmup_iters=WARMUP,
        rep_iters=REPS,
        return_mode="median",
        grad_to_none=grad_to_none,
    )["flops"]
    flops = _attn_flops(
        mask_name, seqlen, num_heads, head_dim_qk, head_dim_v
    )[direction]
    tflops = flops / ms * 1e-9
    work_tflops = flops * 1e-12
    return ms, tflops, work_tflops


IMPLS = ["magi_ffa"]
if CUTEDSL_OK:
    IMPLS.append("flex_cutedsl")
IMPLS.append("torch_flex")
print("Throughput impls:", IMPLS)
print(
    f"S={FLOP_SEQLENS} H={FLOP_HEADS} dims={FLOP_HEAD_DIMS} | "
    "tflops = FLOPs / ms * 1e-9"
)

flop_rows = []
for mask_name in FLOP_MASKS:
    for direction in FLOP_DIRS:
        for seqlen in FLOP_SEQLENS:
            for dq, dk, dv in FLOP_HEAD_DIMS:
                assert dq == dk, "Q/K head_dim must match"
                for impl in IMPLS:
                    label = f"({dq},{dk},{dv})"
                    try:
                        ms, tflops, work_tflops = _bench_impl(
                            impl,
                            mask_name,
                            direction,
                            seqlen,
                            FLOP_HEADS,
                            dq,
                            dv,
                        )
                        flop_rows.append(
                            {
                                "mask": mask_name,
                                "dir": direction,
                                "seqlen": seqlen,
                                "hd": label,
                                "dq": dq,
                                "dv": dv,
                                "impl": impl,
                                "ms": ms,
                                "tflops": tflops,
                                "work_tflops": work_tflops,
                                "status": "ok",
                            }
                        )
                        print(
                            f"{impl:14s} {mask_name:6s} {direction} "
                            f"S={seqlen} hd={label} "
                            f"{ms:7.3f} ms  {tflops:7.1f} TFLOP/s  "
                            f"(work={work_tflops:.3f} TFLOPs)"
                        )
                    except Exception as e:
                        flop_rows.append(
                            {
                                "mask": mask_name,
                                "dir": direction,
                                "seqlen": seqlen,
                                "hd": label,
                                "dq": dq,
                                "dv": dv,
                                "impl": impl,
                                "ms": float("nan"),
                                "tflops": float("nan"),
                                "work_tflops": float("nan"),
                                "status": f"{type(e).__name__}: {e}",
                            }
                        )
                        print(
                            f"{impl:14s} {mask_name:6s} {direction} "
                            f"S={seqlen} hd={label} "
                            f"FAIL: {type(e).__name__}: {e}"
                        )

flop_df = pd.DataFrame(flop_rows)
flop_df[
    ["mask", "dir", "seqlen", "hd", "impl", "ms", "tflops", "work_tflops", "status"]
]



In [ ]:
# TFLOP/s pivots: index=(seqlen, hd), columns=impl
for mask_name in FLOP_MASKS:
    for direction in FLOP_DIRS:
        sub = flop_df[
            (flop_df["mask"] == mask_name) & (flop_df["dir"] == direction)
        ]
        piv = sub.pivot_table(
            index=["seqlen", "hd"], columns="impl", values="tflops"
        )
        ref = "flex_cutedsl" if "flex_cutedsl" in piv.columns else "torch_flex"
        if ref in piv.columns and "magi_ffa" in piv.columns:
            piv["magi/ref"] = piv["magi_ffa"] / piv[ref]
        print(
            f"\n=== TFLOP/s | mask={mask_name} | {direction} "
            f"| H={FLOP_HEADS} (ref={ref}) ==="
        )
        display(piv.style.format("{:.1f}"))



## Notes

- **Correctness** uses eager PyTorch FlexAttention when CuteDSL is unavailable (matches the pytest).
- **Throughput** uses `torch.compile(flex_attention)` so Flex is not unfairly slow vs fused kernels.
- Magi CuteDSL mask_mod signature is `(b, h, q, kv, seqlen_info, aux)`; the notebook wraps torch-style
  `mask_mod(b,h,q,kv)` automatically.
- First Magi FFA call per shape may JIT-compile (slow); subsequent runs reuse the cache.

- Correctness tables are also written to `exps/attn/cutedsl/results/` (`*.md`, `*.html`, `*.csv`). Prefer opening the `.md` or `.html` if Jupyter truncates the display.

